In [3]:
import torch
from torch import nn

device = "mps" if torch.backends.mps.is_available() else "cpu"

torch.manual_seed(42)
n_samples = 2000
n_features = 5

X_normal = torch.randn(n_samples, n_features)
X_normal

tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784],
        [-1.2345, -0.0431, -1.6047, -0.7521,  1.6487],
        [-0.3925, -1.4036, -0.7279, -0.5594, -0.7688],
        ...,
        [-0.6862,  0.1060, -0.1432,  0.5898,  0.1672],
        [ 1.3842, -1.2705, -0.4187, -1.6622, -0.0517],
        [-0.0780, -1.0599,  0.8227, -0.8546,  0.7941]])

In [4]:
X_normal[:, 1] = X_normal[:, 0] * 0.5 + torch.randn(n_samples) * 0.3

X_train, X_test = X_normal[:1600], X_normal[1600:]
X_train, X_test = X_train.to(device), X_test.to(device)

In [8]:
class AnomalyAE(nn.Module):
    def __init__(self, n_features, bottleneck=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 8),
            nn.ReLU(),
            nn.Linear(8, bottleneck)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 8),
            nn.ReLU(),
            nn.Linear(8, n_features)
        )

    def forward(self,  x):
        z = self.encoder(x)
        return self.decoder(z)

model = AnomalyAE(n_features=n_features).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [11]:
epochs = 200

for epoch in range(epochs):
    model.train()
    reconstructed = model(X_train)
    loss = loss_fn(reconstructed, X_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.inference_mode():
        test_recon = model(X_test)
        test_loss = loss_fn(test_recon, X_test)

    if epoch % 20 == 0:
        print(f"Epoch = {epoch}, Train loss: {loss:.5f}, Test loss: {test_loss:.5f}")


Epoch = 0, Train loss: 0.31948, Test loss: 0.32278
Epoch = 20, Train loss: 0.31552, Test loss: 0.32046
Epoch = 40, Train loss: 0.31064, Test loss: 0.31985
Epoch = 60, Train loss: 0.30570, Test loss: 0.31909
Epoch = 80, Train loss: 0.30168, Test loss: 0.31576
Epoch = 100, Train loss: 0.29951, Test loss: 0.31519
Epoch = 120, Train loss: 0.29621, Test loss: 0.31340
Epoch = 140, Train loss: 0.29357, Test loss: 0.31277
Epoch = 160, Train loss: 0.29587, Test loss: 0.31037
Epoch = 180, Train loss: 0.29093, Test loss: 0.30866


In [13]:
def reconstruction_error(model, X):
    model.eval()
    with torch.inference_mode():
        recon = model(X)
        return ((recon-X) ** 2).mean(dim=1)

X_anomaly = torch.randn(50, n_features).to(device)*5
scores_normal = reconstruction_error(model, X_test)
scores_anomaly = reconstruction_error(model, X_anomaly)

print(f"Normal error:  mean={scores_normal.mean():.4f}, max={scores_normal.max():.4f}")
print(f"Anomaly error: mean={scores_anomaly.mean():.4f}, max={scores_anomaly.max():.4f}")

Normal error:  mean=0.3071, max=2.3428
Anomaly error: mean=12.6789, max=34.5525
